
# Advanced ABAC - Group-Based Column Masking

## Data Preparation

This notebook demonstrates group-based column masking
using Customer Support and Finance teams.

Source tables:
- silver_olist.customers
- silver_olist.orders
- silver_olist.order_payments

In [0]:
%sql
SELECT
    o.order_id,
    c.customer_unique_id,
    op.payment_value
FROM course_training_catalog.silver_olist.orders AS o
INNER JOIN course_training_catalog.silver_olist.customers AS c
    ON o.customer_id = c.customer_id
INNER JOIN course_training_catalog.silver_olist.order_payments AS op
    ON o.order_id = op.order_id
LIMIT 20;

order_id,customer_unique_id,payment_value
a9810da82917af2d9aefd1278f1dcfa0,a8b9d3a27068454b1c98cc67d4e31e6f,24.39
25e8ea4e93396b6fa0d3dd708e76c1bd,6f70c0b2f7552832ba46eb57b1c5651e,65.71
42fdf880ba16b47b59251dd489d4441a,4291db0da71914754618cd789aebcd56,128.45
298fcdf1f73eb413e4d26d01b25bc1cd,6e3c218d5f0434ddc4af3d6a60767bbf,96.12
3d7239c394a212faae122962df514ac7,9db67268a3fee1d4b13faaeb90af07c0,51.84
0573b5e23cbd798006520e1d5b4c6714,511c345b0383af6ef25406c564e4d6b6,51.95
d88e0d5fa41661ce03cf6cf336527646,a51f8568095ce89a4ffce365dcb7d46d,188.73
2480f727e869fdeb397244a21b721b67,2faf4a1d502b10555a9f39353ea20148,141.90
8ac09207f415d55acff302df7d6a895c,501598d5821539430a35385474cd44d9,244.15
5a1f6d22f7dfb061ef29216b9af687a1,5986b5f70bf08b414d16a93ac5b1d76c,47.69


In [0]:
%sql
CREATE TABLE IF NOT EXISTS course_training_catalog.silver_olist.abac_department_demo
USING DELTA
AS
SELECT
    o.order_id,
    c.customer_unique_id,
    SUM(op.payment_value) AS payment_value
FROM course_training_catalog.silver_olist.orders AS o
INNER JOIN course_training_catalog.silver_olist.customers AS c
    ON o.customer_id = c.customer_id
INNER JOIN course_training_catalog.silver_olist.order_payments AS op
    ON o.order_id = op.order_id
GROUP BY
    o.order_id,
    c.customer_unique_id;

num_affected_rows,num_inserted_rows


In [0]:
SELECT * 
FROM course_training_catalog.silver_olist.abac_department_demo
LIMIT 20;

order_id,customer_unique_id,payment_value
2e2c60b99754ae1e4d8b18846cfec9f2,74dc6a3e622375fc6aae191555a5a95a,542.66
97c111c4f22dfa1089a9c0b5338dcc79,ae74c314b9c383e5d7bcd99d54ffbc71,44.01
0fba53cb4e9429e38cd2750c32fa24a5,37d35206031031985cbe5577eacbd0f1,52.21
14c0c0bebe2cd50fb1dbdee84fbae03a,43fee0c28d0149eef54898d1d877a3e2,180.55
da46439d7d06cf02cb44d0908ecc3575,9423e765e1167f4c09a153f01ff569b0,41.86
bf951e41c5adb873b7fdcc1e83b406bf,472f97755c020f7fff14b98fc996c900,54.85
1c859fbb6e7f238f3f6134628b845f3a,ae2bc6e3f4ba64afb8405697bc40ab91,26.09
99396be04e138247e9fc0528a62e0919,f3b5ac9b310856ab36153e657e08a495,56.06
e65f1eeee1f52024ad1dcd03447f7482,d5f0cd2eb93b7fc7c2350e758af16dc9,43.28
8e6df87600aae0285919b5529f7f0bfd,bd2c96a76503e424d97c858681ab7cb0,126.48


In [0]:
ALTER TABLE course_training_catalog.silver_olist.abac_department_demo
ALTER COLUMN customer_unique_id
SET TAGS ('abac_data_type' = 'customer_identifier');

In [0]:
ALTER TABLE course_training_catalog.silver_olist.abac_department_demo
ALTER COLUMN payment_value
SET TAGS ('abac_data_type' = 'payment_amount');

In [0]:
%sql
CREATE OR REPLACE FUNCTION
course_training_catalog.silver_olist.mask_customer_by_group(customer_id STRING)
RETURNS STRING
RETURN
    CASE
        WHEN is_account_group_member('customer_support_team')
            THEN customer_id

        WHEN is_account_group_member('finance_team')
            THEN CONCAT(SUBSTRING(customer_id, 1, 4), '****')

        ELSE 'MASKED'
    END;

In [0]:
%sql
CREATE OR REPLACE FUNCTION
course_training_catalog.silver_olist.mask_payment_by_group(amount DOUBLE)
RETURNS DOUBLE
RETURN
    CASE
        WHEN is_account_group_member('finance_team')
            THEN amount

        ELSE CAST(0.0 AS DOUBLE)
    END;

In [0]:
CREATE POLICY mask_customer_identifier_abac
ON TABLE course_training_catalog.silver_olist.abac_department_demo
COLUMN MASK course_training_catalog.silver_olist.mask_customer_by_group
TO `account users`
FOR TABLES
MATCH COLUMNS
    has_tag_value('abac_data_type', 'customer_identifier')
    AS customer_col
ON COLUMN customer_col;

In [0]:
%sql
CREATE POLICY mask_payment_amount_abac
ON TABLE course_training_catalog.silver_olist.abac_department_demo
COLUMN MASK course_training_catalog.silver_olist.mask_payment_by_group
TO `account users`
FOR TABLES
MATCH COLUMNS
    has_tag_value('abac_data_type', 'payment_amount')
    AS payment_col
ON COLUMN payment_col;

In [0]:
SHOW EFFECTIVE POLICIES
ON TABLE course_training_catalog.silver_olist.abac_department_demo;

Policy Name,Policy Type,Catalog,Schema,Table,Comment,on_securable_type,on_securable_fullname
mask_customer_identifier_abac,COLUMN_MASK,course_training_catalog,silver_olist,abac_department_demo,null,TABLE,course_training_catalog.silver_olist.abac_department_demo
mask_payment_amount_abac,COLUMN_MASK,course_training_catalog,silver_olist,abac_department_demo,null,TABLE,course_training_catalog.silver_olist.abac_department_demo
filter_regions_abac,ROW_FILTER,course_training_catalog,silver_olist,null,Allows access to SP and RJ state records using governed tags.,SCHEMA,course_training_catalog.silver_olist


In [0]:

%sql
SELECT
    session_user() AS active_user,
    is_account_group_member('customer_support_team') AS is_support,
    is_account_group_member('finance_team') AS is_finance;

active_user,is_support,is_finance
artificial.intelligence030917@gmail.com,false,false


In [0]:
%sql

SELECT
    order_id,
    customer_unique_id,
    payment_value
FROM course_training_catalog.silver_olist.abac_department_demo
ORDER BY order_id
LIMIT 20;

order_id,customer_unique_id,payment_value
00010242fe8c5a6d1ba2dd792cb16214,MASKED,0.00
00018f77f2f0320c557190d7a144bdd3,MASKED,0.00
00024acbcdf0a6daa1e931b038114c75,MASKED,0.00
00042b26cf59d7ce69dfabb4e55b4fd9,MASKED,0.00
00054e8431b9d7675808bcb819fb4a32,MASKED,0.00
000576fe39319847cbb9d288c5617fa6,MASKED,0.00
0005a1a1728c9d785b8e2b08b904576c,MASKED,0.00
0005f50442cb953dcd1d21e1fb923495,MASKED,0.00
00061f2a7bc09da83e415a52dc8a4af1,MASKED,0.00
00063b381e2406b52ad429470734ebd5,MASKED,0.00
